In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install openai
!pip install pinecone
!pip install pypdf
!pip install tiktoken
!pip install fastapi
!pip install uvicorn
!pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

To use the Pinecone and OpenAI APIs, you'll need API keys. If you don't already have one, create a key for each service.
In Colab, add the keys to the secrets manager under the "🔑" in the left panel. Give them the names `OPENAI_API_KEY` and `PINECONE_API_KEY`.

In [6]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [2]:
from google.colab import files

uploaded = files.upload()

Saving legal_contracts.csv.docx to legal_contracts.csv.docx


In [4]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import Pinecone as LangchainPinecone

from pinecone import Pinecone

# =====================================================
# ADD API KEYS
# =====================================================

OPENAI_API_KEY = "YOUR_OPENAI_KEY"

PINECONE_API_KEY = "YOUR_PINECONE_KEY"

PINECONE_INDEX = "legal-rag-index"

# =====================================================
# LOAD PDF FILES
# =====================================================

documents = []

for filename in uploaded.keys():

    if filename.endswith(".pdf"):

        loader = PyPDFLoader(filename)

        docs = loader.load()

        documents.extend(docs)

print("Documents Loaded:", len(documents))

# =====================================================
# CHUNK DOCUMENTS
# =====================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks Created:", len(chunks))

# =====================================================
# CREATE EMBEDDINGS
# =====================================================

embeddings = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY
)

# =====================================================
# CONNECT PINECONE
# =====================================================

pc = Pinecone(
    api_key=PINECONE_API_KEY
)

# =====================================================
# STORE EMBEDDINGS
# =====================================================

vectorstore = LangchainPinecone.from_documents(
    chunks,
    embeddings,
    index_name=PINECONE_INDEX
)

print("Embeddings uploaded successfully!")

Documents Loaded: 0
Chunks Created: 0


PineconeValueError: No API key provided. Pass api_key='...' or set the PINECONE_API_KEY environment variable.

In [5]:
from langchain_openai import ChatOpenAI

# =====================================================
# LOAD CHAT MODEL
# =====================================================

llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    temperature=0
)

# =====================================================
# ASK QUESTION
# =====================================================

query = "What is the termination clause?"

docs = vectorstore.similarity_search(
    query,
    k=3
)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

prompt = f"""
You are a legal assistant.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

NameError: name 'vectorstore' is not defined